# 01 — Quantum Fundamentals

Welcome to the **Quantum Experience Lab**!  
In this notebook you will:

1. Install Qiskit inside the browser (via `micropip` / Pyodide) — takes ~30 s on first run
2. Create your first qubit and measure it
3. Put a qubit in **superposition** using a Hadamard gate
4. Build a **Bell state** (entangled pair) and measure both qubits
5. (Optional) Send the circuit to the FastAPI backend for real QPU execution

> **Note:** This notebook runs entirely in your browser using [JupyterLite](https://jupyterlite.readthedocs.io/) + [Pyodide](https://pyodide.org/).  
> The Aer high-performance simulator is **not** available in Pyodide; we use `qiskit.primitives.StatevectorSampler` instead.
> For full Aer support, use the backend API via the **Run on QPU** button above.

## Step 0 — Install Qiskit

Run this cell once. It uses `micropip` to install Qiskit from PyPI inside Pyodide.

In [ ]:
import micropip
# pylatexenc is required by Qiskit's matplotlib circuit drawer
await micropip.install(["qiskit==1.2.0", "matplotlib", "pylatexenc"])
print("✅ Qiskit + matplotlib + pylatexenc installed")

## Step 1 — Your first qubit

A **qubit** starts in state |0⟩. Let's create a circuit with one qubit and measure it.

In [ ]:
from qiskit import QuantumCircuit

# 1 qubit, 1 classical bit
qc = QuantumCircuit(1, 1)
qc.measure(0, 0)          # measure qubit 0 → classical bit 0

qc.draw(output="mpl")     # draw the circuit

In [ ]:
from qiskit.primitives import StatevectorSampler

sampler = StatevectorSampler()
job = sampler.run([(qc, None, 1024)])
result = job.result()
counts = dict(result[0].data.c.get_counts())
print("Measurement counts:", counts)
# Expected: {'0': 1024} — qubit always measures |0⟩

## Step 2 — Superposition with the Hadamard gate

The **Hadamard gate** (H) rotates |0⟩ to an equal superposition of |0⟩ and |1⟩:

$$H|0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}$$

After measurement, we expect ~50 % |0⟩ and ~50 % |1⟩.

In [ ]:
qc_h = QuantumCircuit(1, 1)
qc_h.h(0)          # Hadamard gate
qc_h.measure(0, 0)

qc_h.draw(output="mpl")

In [ ]:
import matplotlib.pyplot as plt
from qiskit.visualization import plot_histogram

job_h = sampler.run([(qc_h, None, 1024)])
counts_h = dict(job_h.result()[0].data.c.get_counts())
print("Superposition counts:", counts_h)

plot_histogram(counts_h)

## Step 3 — Entanglement: the Bell state

Apply H to qubit 0, then a **CNOT** (controlled-NOT) gate with qubit 0 as control and qubit 1 as target.
This creates the **Bell state** |Φ⁺⟩:

$$|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$$

When you measure, both qubits are always the **same** — either both 0 or both 1.

In [ ]:
qc_bell = QuantumCircuit(2, 2)
qc_bell.h(0)           # Hadamard on qubit 0
qc_bell.cx(0, 1)       # CNOT: control=0, target=1
qc_bell.measure([0, 1], [0, 1])

qc_bell.draw(output="mpl")

In [ ]:
job_bell = sampler.run([(qc_bell, None, 1024)])
counts_bell = dict(job_bell.result()[0].data.c.get_counts())
print("Bell state counts:", counts_bell)
# Expected: {'00': ~512, '11': ~512} — no '01' or '10'

plot_histogram(counts_bell)

## Step 4 — Export to OpenQASM 3

OpenQASM is the standard interchange format for quantum circuits. The FastAPI backend accepts this format.

In [ ]:
from qiskit.qasm3 import dumps

qasm_str = dumps(qc_bell)
print(qasm_str)

## What's next?

- Click **Run on QPU** in the top bar to send this Bell circuit to a real IBM Quantum backend via the FastAPI service.
- Continue to **notebook 02** to use IBM Bob's AI assistant to explain and optimize circuits.
- Visit the `/demo` route for the full presenter slideshow with animated visualizations.

---
*Quantum Experience Lab — IBM Bob + Qiskit workshop*